In [2]:
# read dataset from 'input.txt' file
with open('input.txt', 'r') as f:
    dataset = f.read()

# split train/test 9/1 
n = len(dataset)
train_data = dataset[:int(n*0.9)]
test_data = dataset[int(n*0.9):]

print("len(dataset):", len(dataset))

len(dataset): 1115394


In [3]:
# basic tokenization
chars = sorted(list(set(dataset))) # get sorted unique characters
vocab_size = len(chars)
print("vocab_size:", vocab_size)

# create mappings from characters to integers and vice versa
stoi = { ch:i for i,ch in enumerate(chars) } # char to int
itos = { i:ch for i,ch in enumerate(chars) } # int to char

def encode(input) -> list[int]:
    return [stoi[c] for c in input]

def decode(input):
    return ''.join([itos[i] for i in input])

print(encode("hii there"))
print(decode(encode("hii there")))



vocab_size: 65
[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [4]:
# dataloader
import torch
batch_size = 4
block_size = 8 # maximum context length for predictions

train_data = torch.tensor(encode(train_data), dtype=torch.long)
test_data = torch.tensor(encode(test_data), dtype=torch.long)

def get_batch(split):
    data = train_data if split == 'train' else test_data
    idx = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in idx])
    y = torch.stack([data[i+1:i+block_size + 1 ] for i in idx])
    return x, y

xb, yb = get_batch('train')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1].tolist()
        target = yb[b, t].item()
        print(f"when input is {context} the target: {target}")

when input is [1] the target: 42
when input is [1, 42] the target: 43
when input is [1, 42, 43] the target: 39
when input is [1, 42, 43, 39] the target: 56
when input is [1, 42, 43, 39, 56] the target: 1
when input is [1, 42, 43, 39, 56, 1] the target: 39
when input is [1, 42, 43, 39, 56, 1, 39] the target: 1
when input is [1, 42, 43, 39, 56, 1, 39, 1] the target: 50
when input is [47] the target: 57
when input is [47, 57] the target: 59
when input is [47, 57, 59] the target: 56
when input is [47, 57, 59, 56] the target: 43
when input is [47, 57, 59, 56, 43] the target: 6
when input is [47, 57, 59, 56, 43, 6] the target: 1
when input is [47, 57, 59, 56, 43, 6, 1] the target: 21
when input is [47, 57, 59, 56, 43, 6, 1, 21] the target: 1
when input is [8] the target: 0
when input is [8, 0] the target: 0
when input is [8, 0, 0] the target: 28
when input is [8, 0, 0, 28] the target: 13
when input is [8, 0, 0, 28, 13] the target: 30
when input is [8, 0, 0, 28, 13, 30] the target: 21
when in

In [22]:
# Create simple BigramLanguageModel
import torch
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        batch_size, block_size, vocab_size = logits.shape
        if targets is None:
            loss = None
        else:
            logits = logits.view(batch_size*block_size, vocab_size)
            targets = targets.view(batch_size*block_size)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
    
model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print("logits shape:", logits.shape) # should be (batch_size*block_size, vocab_size)
print("loss:", loss.item())

    

logits shape: torch.Size([256, 65])
loss: 4.742368221282959


In [23]:
# train model
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
batch_size = 32
for iter in range(10000):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(f"iter {iter}: loss {loss.item()}")


iter 9999: loss 2.5002503395080566


In [26]:
print(decode(model.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=300)[0].tolist()))


TORYRed Ind, be

Ke: s har,
Javered heatrsVO: aleshato te:
NO ar hathr'd m asakers her s, whal go c.

sod Ifisongucond:
Whins d wory te ane t th tm ser r win?

Thin, a:

totheer UFomy hoverttee a aththeayou, VIN he for it we' teset,'s le uthako,
O:
O:
ARoltomyo br A hane army blor Ime ns yomar Hert,


In [30]:
# Simple Self-Attention mechanism (without trainable parameters)
import torch

inputs = torch.tensor([
    [0.43, 0.15, 0.89], # Your
    [0.55, 0.87, 0.66], # journey
    [0.12, 0.34, 0.78], # starts
    [0.22, 0.58, 0.33], # with
    [0.77, 0.25, 0.10], # one
    [0.05, 0.80, 0.55], #step 
])

# goal: how to create intermediate scores for each token?
# calculate socres for "journey" token
input = inputs[1] # [0.55, 0.87, 0.66]
# calculate score by "dot product" with all input tokens
attn_score_2 = torch.empty(inputs.shape[0])
for i, token in enumerate(inputs):
    attn_score_2[i] = torch.dot(input, token)

print("attention scores for 'journey' token:", attn_score_2)


attention scores for 'journey' token: tensor([0.9544, 1.4950, 0.8766, 0.8434, 0.7070, 1.0865])


In [31]:
from torch.nn import functional as F
# stardardize the scores
attn_weight_2 = torch.softmax(attn_score_2, dim=0)
print("standardized attention scores for 'journey' token:", attn_weight_2)

standardized attention scores for 'journey' token: tensor([0.1548, 0.2658, 0.1432, 0.1386, 0.1209, 0.1767])


In [43]:
# compute context vector as weighted sum of input tokens
context_vector_2 = torch.zeros(inputs.shape[1])
for i, token in enumerate(inputs):
    context_vector_2 += attn_weight_2[i] * token
print("context vector for 'journey' token:", context_vector_2)

context vector for 'journey' token: tensor([0.3624, 0.5551, 0.5799])


In [45]:
# now compute for all tokens
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores, dim=-1)
all_context_vectors = attn_weights @ inputs
print("attention scores for all tokens:", attn_weights)
print("context vectors for all tokens:", all_context_vectors)

attention scores for all tokens: tensor([[0.2156, 0.2061, 0.1761, 0.1277, 0.1254, 0.1492],
        [0.1548, 0.2658, 0.1432, 0.1386, 0.1209, 0.1767],
        [0.1904, 0.2062, 0.1796, 0.1389, 0.1108, 0.1741],
        [0.1527, 0.2207, 0.1536, 0.1556, 0.1344, 0.1831],
        [0.1646, 0.2112, 0.1345, 0.1474, 0.2026, 0.1397],
        [0.1478, 0.2331, 0.1595, 0.1517, 0.1055, 0.2024]])
context vectors for all tokens: tensor([[0.3593, 0.4962, 0.6020],
        [0.3624, 0.5551, 0.5799],
        [0.3414, 0.5165, 0.5983],
        [0.3523, 0.5374, 0.5668],
        [0.3985, 0.5021, 0.5365],
        [0.3356, 0.5555, 0.5818]])


In [62]:
# Self-Attention mechanism with trainable parameters
# compute context vector for "journey" token with trainable parameters
x_2 = inputs[1] # [0.55, 0.87, 0.66]
embedding_size = inputs.shape[1]
head_size = 2 # size for query, key, value vectors
W_query = torch.nn.Parameter(torch.rand(embedding_size, head_size), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(embedding_size, head_size), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(embedding_size, head_size), requires_grad=False)

query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

keys = inputs @ W_key
values = inputs @ W_value

attn_scores_2 = query_2 @ keys.T
attn_weights_2 = torch.softmax(attn_scores_2/head_size**0.5, dim=-1)
print("attention scores for 'journey' token with trainable parameters:", attn_weights_2)

context_vector_2 = attn_weights_2 @ values
print("context vector for 'journey' token with trainable parameters:", context_vector_2)

attention scores for 'journey' token with trainable parameters: tensor([0.1855, 0.2263, 0.1555, 0.1360, 0.1429, 0.1539])
context vector for 'journey' token with trainable parameters: tensor([0.4569, 0.5594])


In [ ]:
# all together in a single class
import torch.nn as nn
class SelfAttentionV1(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout): # d_in: input embedding size, d_out: output embedding size
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)
        self.Dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.tril(torch.ones(context_length, context_length), diagonal=0)) # precompute mask for max context length of 1000
    
    def forward(self, x):
        batch_size, num_tokens, d_in = x.shape # x shape: (batch_size, num_tokens, embedding_size)
        querys = self.W_query(x) # shape: (batch_size, num_tokens, d_out)
        keys = self.W_key(x) # shape: (batch_size, num_tokens, d_out)
        values = self.W_value(x) # shape: (batch_size, num_tokens, d_out)
        
        attn_scores = querys @ keys.transpose(1,2) # (batch_size, num_tokens, d_out) @ (batch_size, d_out, num_tokens) --> (batch_size, num_tokens, num_tokens)
        masked_attn_scores = attn_scores.masked_fill(self.mask == 0, float('-inf')) # (batchsize, num_tokens, num_tokens)
        print("masked attention scores for all tokens with trainable parameters:", masked_attn_scores) 
        attn_weights = torch.softmax(masked_attn_scores / (keys.shape[-1] ** 0.5), dim=-1) # (batch_size, num_tokens, num_tokens)
        attn_weights = self.Dropout(attn_weights) # (bath_size, num_tokens, num_tokens)
        print("attention scores for all tokens with trainable parameters:", attn_weights)

        context_vector = attn_weights @ values # (batch_size, num_tokens, num_tokens) @ (batch_size, num_tokens, d_out) --> (batch_size, num_tokens, d_out)
        return context_vector
    
torch.manual_seed(789)
inputs = torch.tensor([
    [0.43, 0.15, 0.89], # Your
    [0.55, 0.87, 0.66], # journey
    [0.12, 0.34, 0.78], # starts
    [0.22, 0.58, 0.33], # with
    [0.77, 0.25, 0.10], # one
    [0.05, 0.80, 0.55], #step 
])
batch = torch.stack([inputs, inputs]) # create a batch of 2 identical sequences
embedding_size = inputs.shape[1]
head_size = 2
model1 = SelfAttentionV1(embedding_size, head_size, batch.shape[1], dropout=0.0)
context_vectors = model1(batch)
print("context vectors for all tokens with trainable parameters:", context_vectors)

masked attention scores for all tokens with trainable parameters: tensor([[[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
         [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
         [0.2694, 0.0745, 0.1388,   -inf,   -inf,   -inf],
         [0.2642, 0.1024, 0.1584, 0.0186,   -inf,   -inf],
         [0.2183, 0.0874, 0.1329, 0.0177, 0.0786,   -inf],
         [0.3408, 0.1270, 0.2005, 0.0198, 0.1290, 0.0078]],

        [[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
         [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
         [0.2694, 0.0745, 0.1388,   -inf,   -inf,   -inf],
         [0.2642, 0.1024, 0.1584, 0.0186,   -inf,   -inf],
         [0.2183, 0.0874, 0.1329, 0.0177, 0.0786,   -inf],
         [0.3408, 0.1270, 0.2005, 0.0198, 0.1290, 0.0078]]],
       grad_fn=<MaskedFillBackward0>)
attention scores for all tokens with trainable parameters: tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
       

In [3]:
import torch
from torch.nn import functional as F
import torch.nn as nn
# implement multihead attention Class
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_lengh, dropout):
        super().__init__()
        print("initializing multi head attention with d_in:", d_in, "d_out:", d_out, "num_heads:", num_heads, "context_length:", context_lengh, "dropout:", dropout)
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_size = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)
        self.Dropout = nn.Dropout(dropout)
        self.OutProj = nn.Linear(d_out, d_out) # output projection to combine heads
        self.register_buffer('mask', torch.tril(torch.ones(context_lengh, context_lengh), diagonal=0)) # precompute mask for max context length of 1000

    def forward(self,x):
        print("input shape:", x.shape)
        batch_size, num_tokens, d_in = x.shape
        querys = self.W_query(x) # shape: (batch_size, num_tokens, d_out = num_heads * head_size)
        keys = self.W_key(x) # shape: (batch_size, num_tokens, d_out = num_heads * head_size)
        values = self.W_value(x) # shape: (batch_size, num_tokens, d_out = num_heads * head_size)
        # print("querys shape:", querys.shape)
        # print("keys shape:", keys.shape)
        # print("values shape:", values.shape)
       
        querys = querys.view(batch_size, num_tokens, self.num_heads, self.head_size).transpose(1,2) # shape: (batch_size, num_heads, num_tokens, head_size)
        keys = keys.view(batch_size, num_tokens, self.num_heads, self.head_size).transpose(1,2) # shape: (batch_size, num_heads, num_tokens, head_size)
        values = values.view(batch_size, num_tokens, self.num_heads, self.head_size).transpose(1,2) # shape: (batch_size, num_heads, num_tokens, head_size)

        attn_scores = querys @ keys.transpose(-2,-1) # (batch_size, num_heads, num_tokens, head_size) @ (batch_size, num_heads, head_size, num_tokens) --> (batch_size, num_heads, num_tokens, num_tokens)
        # print("before masked: ", attn_scores)
        masked_attn_scores = attn_scores.masked_fill(self.mask == 0, float('-inf')) # (batch_size, num_heads, num_tokens, num_tokens)
        # print("mask: ", self.mask)
        # print("after masked: ", masked_attn_scores)
        attn_weights = torch.softmax(masked_attn_scores / (keys.shape[-1] ** 0.5), dim=-1) # (batch_size, num_heads, num_tokens, num_tokens)
        attn_weights = self.Dropout(attn_weights) # (batch_size, num_heads, num_tokens, num_tokens)
        context_vector = attn_weights @ values # (batch_size, num_heads, num_tokens, num_tokens) @ (batch_size, num_heads, num_tokens, head_size) --> (batch_size, num_heads, num_tokens, head_size)

        context_vector = context_vector.transpose(1,2) # (batch_size, num_tokens, num_heads, head_size)
        # print("context: ", context_vector)
        #combine heads
        context_vector = context_vector.contiguous().view(batch_size, num_tokens, self.num_heads * self.head_size) # (batch_size, num_tokens, d_out)
        # print("combined context: ", context_vector)

        context_vector = self.OutProj(context_vector) # (batch_size, num_tokens, d_out)
        return context_vector

        

inputs = torch.tensor([
    [0.43, 0.15, 0.89], # Your
    [0.55, 0.87, 0.66], # journey
    [0.12, 0.34, 0.78], # starts
    [0.22, 0.58, 0.33], # with
    [0.77, 0.25, 0.10], # one
    [0.05, 0.80, 0.55], #step 
])
batch = torch.stack([inputs, inputs]) # create a batch of 4 identical sequences
num_heads = 2
embedding_size = batch.shape[2]
head_size = 2
d_out = num_heads * head_size
context_length = batch.shape[1]
model2 = MultiHeadAttention(embedding_size, d_out, num_heads, context_length, dropout=0.0)

print("multi head attention output shape:", model2(batch)) # should be (batch_size, num_tokens, d_out)


/Users/admin/Documents/life/New_Job/AI/MiniGPT/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


initializing multi head attention with d_in: 3 d_out: 4 num_heads: 2 context_length: 6 dropout: 0.0
input shape: torch.Size([2, 6, 3])
multi head attention output shape: tensor([[[0.4369, 0.1055, 0.2438, 0.0504],
         [0.4946, 0.1616, 0.2320, 0.1354],
         [0.4770, 0.0984, 0.2822, 0.0980],
         [0.4783, 0.0751, 0.2999, 0.0929],
         [0.4766, 0.1003, 0.2651, 0.0962],
         [0.4802, 0.0768, 0.2926, 0.0950]],

        [[0.4369, 0.1055, 0.2438, 0.0504],
         [0.4946, 0.1616, 0.2320, 0.1354],
         [0.4770, 0.0984, 0.2822, 0.0980],
         [0.4783, 0.0751, 0.2999, 0.0929],
         [0.4766, 0.1003, 0.2651, 0.0962],
         [0.4802, 0.0768, 0.2926, 0.0950]]], grad_fn=<ViewBackward0>)
